# N1 — Width × Noise Phase Diagram

**Core contribution of the paper.**

| | |
|---|---|
| Model | CNN5 |
| Dataset | CIFAR-10, n=5000 |
| Full grid | k ∈ {1,2,3,4,6,8,16,32,64} × η ∈ {0%,5%,10%,20%,30%,40%} × 2 seeds |
| Supplement | k=1,3,6 are NEW — run Step 4b if original 72 runs are already done |
| Output | Fig 3 heatmap · Fig 4 DD overlay · Fig 5 phase diagram · benign boundary fit |

## Parallelisation (3 accounts for noise split)

| Account | `MY_NOISE_RATES` | Runs (full) | Runs (supplement only) |
|---------|-----------------|-------------|------------------------|
| **A** | `[0.0, 0.05]` | 18×2=36 | 6×2=12 |
| **B** | `[0.10, 0.20]` | 18×2=36 | 6×2=12 |
| **C** | `[0.30, 0.40]` | 18×2=36 | 6×2=12 |


## Step 1 — Environment

In [ ]:
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Mount Drive + Clone repo

In [ ]:
import os, sys
from google.colab import drive

TOKEN      = 'YOUR_GITHUB_PAT_HERE'  # replace with your token    # ← replace with your PAT
REPO_DIR   = '/content/project-6699'
RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/N1'  # ← shared folder

drive.mount('/content/drive')
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

REPO_URL = f'https://{TOKEN}@github.com/alice20030504/EECS-6699.git'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --branch yixuan {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} checkout yixuan')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Ready:', [f for f in os.listdir('.') if f.endswith('.py')])

## Step 3 — Config
**Each account only changes `MY_NOISE_RATES`.**

In [ ]:
from run_n1 import N1_CONFIG, run_n1, plot_n1
import copy

cfg = copy.deepcopy(N1_CONFIG)

# ── SET YOUR NOISE RATES (one line change per account) ───────────────────────
MY_NOISE_RATES = [0.0, 0.05]       # Account A
# MY_NOISE_RATES = [0.10, 0.20]    # Account B
# MY_NOISE_RATES = [0.30, 0.40]    # Account C
# MY_NOISE_RATES = None            # All noise rates (single account)

active = MY_NOISE_RATES or cfg['noise_rates']
print(f"Noise rates : {[f'{η:.0%}' for η in active]}")
print(f"Widths      : {cfg['widths']}")
print(f"Total runs  : {len(cfg['widths']) * len(active) * len(cfg['seeds'])}")

## Step 4a — Run original 72 runs (k ∈ {2,4,8,16,32,64})
Skip this step if already done — completed runs are auto-skipped.

In [ ]:
import threading, time
def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js; eval_js('0')
        except: pass
threading.Thread(target=_keep_alive, daemon=True).start()

# Original 6 widths only
results = run_n1(cfg, RESULT_DIR,
                 noise_rates=MY_NOISE_RATES,
                 widths=[2, 4, 8, 16, 32, 64],
                 resume=True)
print(f'Done. {len(results)} runs.')

## Step 4b — Supplement: k=1, 3, 6 (NEW points)

Adds:
- **k=1** — underfitting regime, completes the three-stage DD curve
- **k=3** — fills k=2→4 gap, smoother phase boundary
- **k=6** — interpolation threshold at η=15% (matches R1 finding)

36 runs per account, ~4h on T4. Auto-skips if already done.

In [ ]:
results_supp = run_n1(cfg, RESULT_DIR,
                      noise_rates=MY_NOISE_RATES,
                      widths=[1, 3, 6],
                      resume=True)
print(f'Supplement done. {len(results_supp)} runs.')

## Step 5 — Plot (run after ALL accounts finish)

Generates:
- **Fig 3** — test error heatmap
- **Fig 4** — DD curves overlay (one per noise level)
- **Fig 5** — three-region phase diagram (per-column baseline, Δ<5%/15%)
- **Fig 5b** — empirical benign boundary fit w_benign ≈ c·η^α

Uses all JSONs present in RESULT_DIR — partial results produce partial plots.

In [ ]:
plot_n1(RESULT_DIR, cfg)

from IPython.display import Image, display
from pathlib import Path
for fig_name in ['fig3_n1_heatmap.png', 'fig4_n1_dd_overlay.png',
                 'fig5_n1_phase_diagram.png', 'fig5b_benign_boundary.png']:
    p = Path(RESULT_DIR) / fig_name
    if p.exists():
        print(f'\n--- {fig_name} ---')
        display(Image(str(p)))

## Step 6 — Summary table

In [ ]:
import pandas as pd, json
from src.io_utils import load_results
from pathlib import Path

results = load_results(RESULT_DIR, pattern='n1_*.json')
df = pd.DataFrame([{
    'k':         r['width_multiplier'],
    'eta':       f"{r['noise_rate']:.0%}",
    'seed':      r['seed'],
    'n_params':  r['n_params'],
    'train_err': f"{r['train_error']:.3f}",
    'test_err':  f"{r['test_error']:.3f}",
} for r in results]).sort_values(['k', 'eta', 'seed'])
print(f'Total runs in folder: {len(df)}')
print(df.to_string(index=False))

fit_path = Path(RESULT_DIR) / 'benign_boundary_fit.json'
if fit_path.exists():
    with open(fit_path) as f:
        fit = json.load(f)
    print(f"\nBenign boundary: {fit['formula']}")